In [ ]:
!pip uninstall -y whisper
!pip install -q -U openai-whisper
!apt-get install -y -qq ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 18.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
from pathlib import Path
import whisper

video_path = Path("/content/drive/MyDrive/mock video/1.mp4")

model = whisper.load_model("large")

result = model.transcribe(
    str(video_path),  # chuyển PosixPath thành chuỗi
    language="vi"
)

print(result["text"])

100%|█████████████████████████████████████| 2.88G/2.88G [00:40<00:00, 75.5MiB/s]


 Hãy subscribe cho kênh Ghiền Mì Gõ Để không bỏ lỡ những video hấp dẫn Hãy subscribe cho kênh Ghiền Mì Gõ Để không bỏ lỡ những video hấp dẫn Hãy subscribe cho kênh Ghiền Mì Gõ Để không bỏ lỡ những video hấp dẫn Hãy subscribe cho kênh Ghiền Mì Gõ Để không bỏ lỡ những video hấp dẫn Hãy subscribe cho kênh Ghiền Mì Gõ Để không bỏ lỡ những video hấp dẫn Hãy subscribe cho kênh Ghiền Mì Gõ Để không bỏ lỡ những video hấp dẫn Hãy subscribe cho kênh Ghiền Mì Gõ Để không bỏ lỡ những video hấp dẫn Hãy subscribe cho kênh Ghiền Mì Gõ Để không bỏ lỡ những video hấp dẫn Hãy subscribe cho kênh Ghiền Mì Gõ Để không bỏ lỡ những video hấp dẫn Hãy subscribe cho kênh Ghiền Mì Gõ Để không bỏ lỡ những video hấp dẫn Hãy subscribe cho kênh Ghiền Mì Gõ Để không bỏ lỡ những video hấp dẫn Hãy subscribe cho kênh Ghiền Mì Gõ Để không bỏ lỡ những video hấp dẫn Hãy subscribe cho kênh Ghiền Mì Gõ Để không bỏ lỡ những video hấp dẫn Hãy subscribe cho kênh Ghiền Mì Gõ Để không bỏ lỡ những video hấp dẫn Hãy subscribe cho k

In [ ]:
# ============================================================
# ONE-CELL PHOWHISPER-LARGE: MP4 -> WAV -> VIETNAMESE TRANSCRIPT
# (Optimized for T4 GPU utilization)
# ============================================================

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!pip install -q -U "transformers==4.46.3" accelerate librosa soundfile
!apt-get update -qq
!apt-get install -y -qq ffmpeg

import subprocess
import torch
from transformers import (
    AutoModelForSpeechSeq2Seq,
    AutoProcessor,
    pipeline,
)

# ------------------------------------------------------------
# ONLY EDIT THIS PATH
# ------------------------------------------------------------
VIDEO_PATH = "/content/drive/MyDrive/mock video/1.mp4"

MODEL_ID = "vinai/PhoWhisper-large"
AUDIO_PATH = "/content/phowhisper_audio.wav"

# ------------------------------------------------------------
# 1. Validate input
# ------------------------------------------------------------
if not VIDEO_PATH.strip():
    raise ValueError(
        'VIDEO_PATH is empty. Example:\n'
        'VIDEO_PATH = "/content/drive/MyDrive/mock video/1.mp4"'
    )

VIDEO_PATH = os.path.abspath(VIDEO_PATH)

if not os.path.isfile(VIDEO_PATH):
    raise FileNotFoundError(f"Video does not exist: {VIDEO_PATH}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU not found. Select Runtime → Change runtime type → GPU."
    )

torch.backends.cudnn.benchmark = True  # speeds up repeated fixed-size conv ops

gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name} ({gpu_mem_gb:.1f} GB VRAM)")
print("Video:", VIDEO_PATH)

# ------------------------------------------------------------
# 2. Extract MP4 audio to mono 16-kHz WAV
# ------------------------------------------------------------
print("\nExtracting audio from MP4...")

ffmpeg_result = subprocess.run(
    [
        "ffmpeg",
        "-y",
        "-i", VIDEO_PATH,
        "-map", "0:a:0",
        "-vn",
        "-ac", "1",
        "-ar", "16000",
        "-c:a", "pcm_s16le",
        AUDIO_PATH,
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)

if ffmpeg_result.returncode != 0:
    raise RuntimeError(
        "FFmpeg could not extract audio. The video may not contain an "
        f"audio stream.\n\n{ffmpeg_result.stderr[-2000:]}"
    )

if not os.path.isfile(AUDIO_PATH) or os.path.getsize(AUDIO_PATH) == 0:
    raise RuntimeError("FFmpeg created an empty WAV file.")

print("Audio created:", AUDIO_PATH)

# ------------------------------------------------------------
# 3. Load PhoWhisper-large (with fast attention + full GPU placement)
# ------------------------------------------------------------
print("\nLoading PhoWhisper-large...")

dtype = torch.float16

processor = AutoProcessor.from_pretrained(MODEL_ID)

# Try the fast SDPA attention kernel first (well supported on T4);
# fall back to eager if the installed transformers/torch combo doesn't support it.
try:
    model = AutoModelForSpeechSeq2Seq.from_pretrained(
        MODEL_ID,
        torch_dtype=dtype,
        low_cpu_mem_usage=True,
        use_safetensors=False,
        attn_implementation="sdpa",
    ).to("cuda").eval()
    print("Using SDPA attention (fast path).")
except Exception as e:
    print(f"SDPA unavailable ({e}), falling back to eager attention.")
    model = AutoModelForSpeechSeq2Seq.from_pretrained(
        MODEL_ID,
        torch_dtype=dtype,
        low_cpu_mem_usage=True,
        use_safetensors=False,
    ).to("cuda").eval()

print(f"GPU memory after model load: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# ------------------------------------------------------------
# 4. Build pipeline with a batch size sized for T4's 16GB,
#    with automatic fallback if you hit an OOM.
# ------------------------------------------------------------
def build_transcriber(batch_size):
    return pipeline(
        task="automatic-speech-recognition",
        model=model,
        tokenizer=processor.tokenizer,
        feature_extractor=processor.feature_extractor,
        torch_dtype=dtype,
        device=0,
        chunk_length_s=30,
        stride_length_s=(5, 5),
        batch_size=batch_size,
    )

BATCH_SIZE = 24  # T4 (16GB) can usually push higher than 16 for a large model in fp16
transcriber = build_transcriber(BATCH_SIZE)

# ------------------------------------------------------------
# 5. Transcribe WAV — not the original MP4 — with OOM auto-retry
# ------------------------------------------------------------
print(f"\nTranscribing (batch_size={BATCH_SIZE})...")

try:
    result = transcriber(
        AUDIO_PATH,
        return_timestamps=True,
        generate_kwargs={"language": "vi", "task": "transcribe"},
    )
except torch.cuda.OutOfMemoryError:
    print("OOM at batch_size", BATCH_SIZE, "-> retrying at batch_size=8")
    torch.cuda.empty_cache()
    BATCH_SIZE = 8
    transcriber = build_transcriber(BATCH_SIZE)
    result = transcriber(
        AUDIO_PATH,
        return_timestamps=True,
        generate_kwargs={"language": "vi", "task": "transcribe"},
    )

print(f"Peak GPU memory used: {torch.cuda.max_memory_allocated()/1e9:.2f} GB "
      f"/ {gpu_mem_gb:.1f} GB total")

# ------------------------------------------------------------
# 6. Display results
# ------------------------------------------------------------
print("\n========== FULL TRANSCRIPT ==========\n")
print(result["text"].strip())

print("\n========== TIMESTAMPS ==========\n")

for chunk in result.get("chunks", []):
    start, end = chunk["timestamp"]
    start_text = f"{start:.2f}" if start is not None else "?"
    end_text = f"{end:.2f}" if end is not None else "?"
    print(f"[{start_text}s → {end_text}s] {chunk['text'].strip()}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 93.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 85.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
GPU: Tesla T4 (15.6 GB VRAM)
Video: /content/drive/MyDrive/mock video/1.mp4

Extracting audio from MP4...
Audio created: /content/phowhisper_audio.wav

Loading PhoWhisper-large...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/339 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/805 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/6.17G [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

Using SDPA attention (fast path).
GPU memory after model load: 3.09 GB

Transcribing (batch_size=24)...


/usr/local/lib/python3.12/dist-packages/transformers/models/whisper/generation_whisper.py:509: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
You have passed task=transcribe, but also have set `forced_decoder_ids` to [[1, None], [2, 50359]] which creates a conflict. `forced_decoder_ids` will be ignored in favor of task=transcribe.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Peak GPU memory used: 10.91 GB / 15.6 GB total

========== FULL TRANSCRIPT ==========

đăng ký và chia sẻ nhé.chào mừng quý vị đến với chương trình sáu mươi giây của đài truyền hình thành phố hồ chí minh chương trình sách này có những thông tin nổi bật sau đây.đồng băng sông cửu long với tình trạng sụt lún gấp gần hai mươi lần so với nước biển dần vẫn chuyển các tốc trái tim từ hà nội về huế ghép cho bệnh nhân châu âu trúng chọi với nhiệt độ nóng như thiêu đốt cùng những đám cháy rực sụt lún đang là vấn đề cấp bách với đồng bằng sông cửu long khi có nơi sụt lún trung bình lên tới năm phẩy bảy xăngtimét một năm tức là gấp gần hai mươi lần so với nước biển dần dự báo phần lớn diện tích có thể sẽ nằm dưới mực nước biển trung bình vào cuối thế kỷ hai mươi mốt chiều ngày ba mươi mốt tháng bảy tại hà nội cơ quan chuyên môn đã báo cáo lãnh đạo bộ nông nghiệp và phát triển nông thôn về đề án tổng thể phòng chống sụt lún đất sạt lở bờ sông bờ biển ngập úng hạn hán sáp nhập mặn tại đồng bằng sôn

## Đánh giá

Kết quả tốt, ít lỗi chính tả, bắt được phần lớn từ.
Tuy nhiên chạy hơi lâu và cần phải làm sách trước khi embed.